# Monitoring: the out-of-time test on 2023

This notebook tests the final model of `03_glm.ipynb` on the 2023 rows, which no earlier step used. It follows the same
yardstick as the GLM notebook (Tweedie deviance, Gini, lift and A/E) and uses aggregates only.

**The frozen model.** Two predictions are scored for every person-year. The **technical model** is the final structure, with
9 rating factors, the analysis-only factors priced at their reference level, adult age as a natural spline and the base rate
rebalanced on 2022 train. The **tariff** is its surrogate model (tariff) with five-year age bands, which is what would be
charged. Both are fitted on the 2022 train split only. The 2023 rows are scored and are never used to fit anything, except
where a later section says so and labels it as a comparison. If the model changes later, this notebook is rerun.

**Prices.** Costs of 2023 are as recorded. Costs of 2022 are on-levelled to 2023 prices with the PCE-Health index, so the two
years are on one price basis.

## Frozen model and the 2023 scoring frame

In [1]:
import pandas as pd

from health_pricing.models import glm, outofsample

pd.options.display.float_format = "{:,.3f}".format
frozen = outofsample.freeze()
oot = outofsample.load_out_of_time(frozen)
pd.DataFrame(
    {
        "model": [outofsample.TECHNICAL, outofsample.TARIFF],
        "parameters": [
            frozen.spline.model.parameters,
            frozen.final_tariff.tariff.parameters,
        ],
        "rebalancing_factor": [frozen.balance, 1.0],
    }
)

,model,parameters,rebalancing_factor
0,(a) Technical model,64,1.116
1,(a) Tariff,26,1.000


### 2023 next to 2022

The table puts 2023 next to 2022 train and test. Cost per exposure year is on the 2023 price basis. The last columns show the
cost above the large-claim cap of 100,000 dollars per person-year, which the provisional 6.5 percent load covers.

In [2]:
period_summary = outofsample.period_summary(frozen, oot)
outofsample.save_table(period_summary, "period_summary.csv")
period_summary

,period,person_years,exposure,claimants,claimant_rate,cost_per_exposure_year,mean_cost_per_claimant,claimants_above_cap,share_of_cost_above_cap,excess_load,cost_per_exposure_year_vs_2022_train
0,2022 train,17596,"17,440.750",7195,0.409,"2,183.395","5,292.570",32,0.061,0.065,0.000
1,2022 test,4151,"4,100.333",1725,0.416,"2,377.607","5,651.584",11,0.108,0.121,0.089
2,2023,18463,"18,301.250",7763,0.420,"2,551.995","6,016.320",46,0.058,0.062,0.169


The 2023 rows have 18,463 person-years and 18,301 years of exposure. Cost per exposure year is 2,552 dollars, which is 16.9
percent above 2022 train (2,183) and 7.3 percent above 2022 test (2,378). The claimant rate is 42.0 percent against 40.9
percent in train and 41.6 percent in test, and the mean cost per claimant is 6,016 dollars against 5,293 in train. So both
more people claim and each claim costs more, and the next section sorts out how much of each is the model's mix and how much
is a change in level. A total of 46 person-years in 2023 are above the cap, and the excess is 6.2 percent of capped cost,
against the provisional load of 6.5 percent.

### Levels that are new in 2023

Every 2023 level of every factor is mapped to a group with the M3 grouping. A level that the grouping had not seen would be
treated as the reference group and listed here.

In [3]:
outofsample.save_table(oot.new_levels, "new_levels_2023.csv")
print(f"Levels new in 2023: {len(oot.new_levels)}")
oot.new_levels

Levels new in 2023: 0


,factor,level,rows,treated_as


No level in 2023 is new to the grouping, so no row is moved to a reference group.

### The frozen model reproduces the GLM notebook

The technical model and the tariff are rebuilt from the 2022 train split. Their scores on train and test should equal those
saved by the GLM notebook (a test checks this to rounding).

In [4]:
scores = pd.concat(
    [
        glm.candidate_scores(
            name,
            parameters,
            {
                "train": (frozen.prepared.train, predictor(frame_train)),
                "test": (frozen.prepared.test, predictor(frame_test)),
            },
        )
        for name, parameters, predictor, frame_train, frame_test in (
            (
                outofsample.TECHNICAL,
                frozen.spline.model.parameters,
                frozen.technical,
                frozen.final_tariff.frames["spline"].train,
                frozen.final_tariff.frames["spline"].test,
            ),
            (
                outofsample.TARIFF,
                frozen.final_tariff.tariff.parameters,
                frozen.tariff,
                frozen.prepared.train,
                frozen.prepared.test,
            ),
        )
    ],
    ignore_index=True,
)
outofsample.save_table(scores.drop(columns="dropped_reason"), "frozen_model_scores_2022.csv")
scores

,candidate,split,tweedie_deviance,mean_tweedie_deviance,tweedie_deviance_p1_3,tweedie_deviance_p1_7,gini,a_e,parameters,dropped_reason
0,(a) Technical model,train,"3,031,554.108",172.287,"12,879,619.363","887,948.424",0.492,1.000,64,
1,(a) Technical model,test,"831,786.305",200.382,"3,671,617.500","231,890.463",0.386,1.081,64,
2,(a) Tariff,train,"3,051,674.804",173.430,"12,968,757.123","892,816.020",0.488,1.000,26,
3,(a) Tariff,test,"838,173.103",201.921,"3,700,064.950","233,355.176",0.378,1.081,26,


The technical model has a test deviance of 831,786 and a Gini of 0.386, and the tariff has 838,173 and 0.378. Both equal the
values saved by the GLM notebook. On train, predicted cost equals actual cost by construction (A/E of 1.000), and on the test
split A/E is 1.081.